# LeetCode 2285 — Maximum Total Importance of Roads bằng Python

Link đề: https://leetcode.com/problems/maximum-total-importance-of-roads/ · Độ khó: **Medium** · Chủ đề: Greedy, Graph, Sorting, Heap (Priority Queue)

Skeleton LeetCode:

```python
class Solution:
    def maximumImportance(self, n: int, roads: List[List[int]]) -> int:
        ...
```

## 1. Mục tiêu học

1. **Đổi góc nhìn tổng:** tổng theo **cạnh** `Σ (val[a] + val[b])` bằng tổng theo **đỉnh** `Σ deg[v] · val[v]` — kỹ thuật "đếm đóng góp" (contribution technique).
2. Nhận ra bài toán còn lại là **ghép hai dãy** để tích vô hướng lớn nhất ⇒ **bất đẳng thức sắp xếp lại** (rearrangement inequality): ghép lớn với lớn.
3. Chứng minh tham lam bằng **lập luận hoán đổi** (exchange argument).
4. Cài đặt O(n log n) bằng sắp xếp, và O(n + m) bằng **đếm phân phối** (counting sort) vì bậc ≤ n − 1.
5. Kiểm chứng bằng vét cạn mọi hoán vị trên đồ thị nhỏ ngẫu nhiên, và thấy khi nào tham lam **không còn đúng** (hàm mục tiêu không tuyến tính).

## 2. Đề bài & ví dụ

Có `n` thành phố đánh số `0 … n−1` và danh sách đường hai chiều `roads[i] = [aᵢ, bᵢ]`. Gán cho mỗi thành phố một giá trị nguyên **khác nhau** trong `1 … n` (mỗi giá trị dùng đúng một lần). **Độ quan trọng** của một con đường = tổng giá trị của hai thành phố nó nối. Trả về **tổng độ quan trọng lớn nhất** của mọi con đường.

**Ví dụ 1:** `n = 5`, `roads = [[0,1],[1,2],[2,3],[0,2],[1,3],[2,4]]` → `43` (gán `[2,4,5,3,1]`).

**Ví dụ 2:** `n = 5`, `roads = [[0,3],[2,4],[1,3]]` → `20` (gán `[4,3,2,5,1]`).

**Ràng buộc:** `2 ≤ n ≤ 5·10⁴`; `1 ≤ len(roads) ≤ 5·10⁴`; `aᵢ ≠ bᵢ`; không có đường trùng.

In [1]:
from typing import List
from itertools import permutations
from collections import Counter
import heapq, random, time

examples = [(5, [[0, 1], [1, 2], [2, 3], [0, 2], [1, 3], [2, 4]], 43), (5, [[0, 3], [2, 4], [1, 3]], 20)]
for n, roads, exp in examples:
    print(f"n={n} roads={roads} -> kỳ vọng {exp}")

def importance(roads, val):
    return sum(val[a] + val[b] for a, b in roads)

print("kiểm tra phép gán của đề:", importance(examples[0][1], [2, 4, 5, 3, 1]), importance(examples[1][1], [4, 3, 2, 5, 1]))

n=5 roads=[[0, 1], [1, 2], [2, 3], [0, 2], [1, 3], [2, 4]] -> kỳ vọng 43
n=5 roads=[[0, 3], [2, 4], [1, 3]] -> kỳ vọng 20
kiểm tra phép gán của đề: 43 20


## 3. Kiến thức nền

### 3.1 Bậc của đỉnh và bổ đề bắt tay

`deg[v]` = số đường có đầu mút `v`. Mỗi đường góp 1 vào bậc của **hai** đầu mút ⇒ `Σ deg[v] = 2m` (handshake lemma).

In [2]:
def degrees(n, roads):
    deg = [0] * n
    for a, b in roads:
        deg[a] += 1; deg[b] += 1
    return deg

for n, roads, _ in examples:
    d = degrees(n, roads); print(d, "tổng =", sum(d), "= 2m =", 2 * len(roads))

[2, 3, 4, 2, 1] tổng = 12 = 2m = 12
[1, 1, 1, 2, 1] tổng = 6 = 2m = 6


### 3.2 Đếm đóng góp: đổi thứ tự lấy tổng

`Σ_{(a,b)} (val[a] + val[b])` — giá trị `val[v]` xuất hiện **một lần cho mỗi đường chạm `v`** ⇒ tổng = `Σ_v deg[v] · val[v]`.

In [3]:
roads = examples[0][1]; val = [2, 4, 5, 3, 1]; d = degrees(5, roads)
print("theo cạnh:", importance(roads, val), "| theo đỉnh:", sum(dv * vv for dv, vv in zip(d, val)))

theo cạnh: 43 | theo đỉnh: 43


### 3.3 Bất đẳng thức sắp xếp lại (rearrangement inequality)

Cho `x₁ ≤ … ≤ xₙ` và `y₁ ≤ … ≤ yₙ`. Với mọi hoán vị `σ`: `Σ xᵢ y_{σ(i)} ≤ Σ xᵢ yᵢ` (ghép **cùng chiều** lớn nhất) và `≥ Σ xᵢ y_{n+1−i}` (ghép **ngược chiều** nhỏ nhất).

In [4]:
x = [1, 2, 2, 5]; y = [1, 2, 3, 4]
vals = sorted(sum(a * b for a, b in zip(x, p)) for p in permutations(y))
print("min =", vals[0], "(ngược chiều:", sum(a * b for a, b in zip(x, y[::-1])), ") | max =", vals[-1],
      "(cùng chiều:", sum(a * b for a, b in zip(x, y)), ")")

min = 19 (ngược chiều: 19 ) | max = 31 (cùng chiều: 31 )


### 3.4 `enumerate(…, 1)` và `sum` với generator

In [5]:
print(list(enumerate([3, 1, 2], 1)), sum(i * v for i, v in enumerate(sorted([3, 1, 2]), 1)))

[(1, 3), (2, 1), (3, 2)] 14


### 3.5 Đếm phân phối (counting sort) khi khóa nhỏ

Bậc nằm trong `0 … n−1` ⇒ đếm số đỉnh có mỗi bậc rồi duyệt bậc tăng dần, không cần sắp xếp so sánh.

In [6]:
d = [3, 3, 4, 2, 1]
cnt = [0] * len(d)
for v in d: cnt[v] += 1
print("cnt theo bậc:", cnt, "-> dãy tăng dần:", [b for b in range(len(cnt)) for _ in range(cnt[b])])

cnt theo bậc: [0, 1, 1, 2, 1] -> dãy tăng dần: [1, 2, 3, 3, 4]


## 4. Ý tưởng nền tảng / trực giác

**Bước 1 — đổi tổng:** `Tổng = Σ_v deg[v] · val[v]`. Cấu trúc đồ thị chỉ còn ảnh hưởng qua **dãy bậc**; ai nối với ai không quan trọng nữa.

**Bước 2 — ghép:** ta có `n` "hệ số" `deg[v]` và `n` "giá trị" `1 … n`. Đỉnh bậc cao "nhân" giá trị của nó nhiều lần ⇒ nên nhận giá trị lớn. Theo bất đẳng thức sắp xếp lại: sắp `deg` tăng dần, gán `1, 2, …, n` theo thứ tự đó.

Tính tay ví dụ 1: `deg = [2, 3, 4, 2, 1]`.

| thứ hạng (bậc tăng) | đỉnh | bậc | giá trị | đóng góp |
|---|---|---|---|---|
| 1 | 4 | 1 | 1 | 1 |
| 2 | 0 | 2 | 2 | 4 |
| 3 | 3 | 2 | 3 | 6 |
| 4 | 1 | 3 | 4 | 12 |
| 5 | 2 | 4 | 5 | 20 |

Tổng `1 + 4 + 6 + 12 + 20 = 43` ✔ — và phép gán `val = [2, 4, 5, 3, 1]` đúng là phép gán của đề.

Ví dụ 2: `deg = [1, 1, 1, 2, 1]` ⇒ sắp xếp `[1,1,1,1,2]` ⇒ `1+2+3+4+2·5 = 20` ✔.

**Hòa bậc:** hai đỉnh cùng bậc đổi giá trị cho nhau không đổi tổng ⇒ nhiều phép gán tối ưu (đề chỉ hỏi giá trị tổng).

In [7]:
def assignment(n, roads):
    d = degrees(n, roads)
    order = sorted(range(n), key=lambda v: d[v])
    val = [0] * n
    for rank, v in enumerate(order, 1): val[v] = rank
    return val

for n, roads, exp in examples:
    v = assignment(n, roads); print("gán:", v, "-> tổng", importance(roads, v), "| kỳ vọng", exp)

gán: [2, 4, 5, 3, 1] -> tổng 43 | kỳ vọng 43
gán: [1, 2, 3, 5, 4] -> tổng 20 | kỳ vọng 20


## 5. Cách 1 — Brute Force / baseline

Thử **mọi hoán vị** giá trị `1 … n` — O(n! · m). Chỉ dùng để kiểm chứng với `n ≤ 8`.

In [8]:
def brute(n, roads):
    best = 0
    for perm in permutations(range(1, n + 1)):
        best = max(best, sum(perm[a] + perm[b] for a, b in roads))
    return best

for n, roads, exp in examples:
    got = brute(n, roads); print(got, "OK" if got == exp else "SAI")

43 OK
20 OK


### 5.1 Baseline 2 — vét cạn trên dãy bậc (n! · n) thay vì trên cạnh

Dùng công thức theo đỉnh: mỗi hoán vị chỉ tốn O(n) thay vì O(m). Vẫn giai thừa, nhưng cho thấy bài thực chất chỉ phụ thuộc vào dãy bậc.

In [9]:
def brute_deg(n, roads):
    d = degrees(n, roads)
    return max(sum(dv * pv for dv, pv in zip(d, p)) for p in permutations(range(1, n + 1)))

for n, roads, exp in examples:
    print(brute_deg(n, roads), "OK" if brute_deg(n, roads) == exp else "SAI")
t0 = time.perf_counter(); brute_deg(8, [[i, (i + 1) % 8] for i in range(8)]); print(f"n=8: {(time.perf_counter()-t0)*1000:.0f} ms — n=12 sẽ chậm hơn ~12·11·10·9 = 11880 lần")

43 OK
20 OK
n=8: 19 ms — n=12 sẽ chậm hơn ~12·11·10·9 = 11880 lần


**Nút thắt:** không gian hoán vị khổng lồ, nhưng phần lớn hoán vị rõ ràng tệ hơn: nếu đỉnh bậc thấp có giá trị lớn hơn đỉnh bậc cao, hoán đổi luôn không làm tổng giảm. Lặp hoán đổi cho tới khi "bậc tăng ⇒ giá trị tăng" ⇒ chỉ còn **một** dạng phép gán cần xét — tham lam.

## 6. Ý tưởng thuật toán / bước đột phá

1. Tính `deg` (O(n + m)).
2. Sắp xếp `deg` tăng dần.
3. Trả về `Σ_{i=1..n} i · deg_sorted[i]`.

**Lập luận hoán đổi (ngắn):** giả sử phép gán tối ưu có hai đỉnh `u, v` với `deg[u] < deg[v]` nhưng `val[u] > val[v]`. Đổi giá trị của chúng: tổng thay đổi
`Δ = deg[u]·val[v] + deg[v]·val[u] − deg[u]·val[u] − deg[v]·val[v] = (deg[v] − deg[u]) · (val[u] − val[v]) > 0`
— mâu thuẫn với tối ưu. Vậy trong mọi phép gán tối ưu, bậc lớn hơn **thực sự** ⇒ giá trị lớn hơn. Các đỉnh cùng bậc đổi giá trị tùy ý không đổi tổng ⇒ tổng tối ưu = tích vô hướng của `deg` đã sắp với `1 … n`.

**Điều quan trọng:** cạnh không trùng và không có khuyên không ảnh hưởng thuật toán; nếu có cạnh trùng, công thức vẫn đúng (bậc đếm cả bội) — xem 15.2.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Sắp xếp bậc (cách doocs)

In [10]:
def sort_degrees(n, roads):
    deg = [0] * n
    for a, b in roads:
        deg[a] += 1; deg[b] += 1
    deg.sort()
    return sum(i * v for i, v in enumerate(deg, 1))

for n, roads, exp in examples:
    print(sort_degrees(n, roads), "OK" if sort_degrees(n, roads) == exp else "SAI")

43 OK
20 OK


### 7.2 Cách 3 — Đếm phân phối theo bậc: O(n + m), không sắp xếp so sánh

Với mỗi bậc `b` có `c` đỉnh, các đỉnh này nhận một **khối giá trị liên tiếp** `k+1 … k+c` ⇒ đóng góp `b · (tổng cấp số cộng)` = `b · c · (2k + c + 1) / 2`.

In [11]:
def counting(n, roads):
    deg = [0] * n
    for a, b in roads:
        deg[a] += 1; deg[b] += 1
    cnt = [0] * n                       # bậc tối đa n - 1
    for d in deg: cnt[d] += 1
    total = 0; k = 0                    # k = số giá trị đã dùng
    for b in range(n):
        c = cnt[b]
        if c:
            total += b * (c * (2 * k + c + 1) // 2)
            k += c
    return total

for n, roads, exp in examples:
    print(counting(n, roads), "OK" if counting(n, roads) == exp else "SAI")

43 OK
20 OK


`c · (2k + c + 1)` luôn chẵn (một trong `c`, `2k + c + 1` là chẵn) ⇒ phép chia nguyên chính xác.

### 7.3 Cách 4 — Heap: lấy dần đỉnh bậc lớn nhất, gán `n, n−1, …`

Không tốt hơn sắp xếp, nhưng là mẫu "phát giá trị lớn nhất cho ứng viên tốt nhất" quen thuộc — tiện khi giá trị đến dần (dòng dữ liệu).

In [12]:
def heap_assign(n, roads):
    deg = degrees(n, roads)
    h = [-d for d in deg]; heapq.heapify(h)
    total = 0
    for value in range(n, 0, -1):
        total += value * -heapq.heappop(h)
    return total

for n, roads, exp in examples:
    print(heap_assign(n, roads), "OK" if heap_assign(n, roads) == exp else "SAI")

43 OK
20 OK


### 7.4 Cách 5 — Dùng `Counter` trên danh sách đầu mút (một dòng)

Lưu ý: `Counter` **bỏ qua** đỉnh cô lập (bậc 0) — cần bù số 0 để các giá trị nhỏ nhất dành cho chúng.

In [13]:
def counter_oneline(n, roads):
    c = Counter(x for r in roads for x in r)
    deg = sorted(c.values())
    offset = n - len(deg)                        # số đỉnh bậc 0 nhận các giá trị 1..offset
    return sum(i * v for i, v in enumerate(deg, offset + 1))

for n, roads, exp in examples:
    print(counter_oneline(n, roads), "OK" if counter_oneline(n, roads) == exp else "SAI")
print("đồ thị có đỉnh cô lập:", counter_oneline(6, [[0, 1]]), "vs", sort_degrees(6, [[0, 1]]))

43 OK
20 OK
đồ thị có đỉnh cô lập: 11 vs 11


## 8. TRACE từng bước

### 8.1 Trace cách 2 trên ví dụ 1

In [14]:
def trace(n, roads):
    deg = [0] * n
    for a, b in roads:
        deg[a] += 1; deg[b] += 1
        print(f"đường ({a},{b}) -> deg = {deg}")
    order = sorted(range(n), key=lambda v: deg[v])
    total = 0
    for i, v in enumerate(order, 1):
        total += i * deg[v]
        print(f"  giá trị {i} -> đỉnh {v} (bậc {deg[v]}): +{i * deg[v]:>3}  tổng = {total}")
    return total

print("kết quả:", trace(*examples[0][:2]))

đường (0,1) -> deg = [1, 1, 0, 0, 0]
đường (1,2) -> deg = [1, 2, 1, 0, 0]
đường (2,3) -> deg = [1, 2, 2, 1, 0]
đường (0,2) -> deg = [2, 2, 3, 1, 0]
đường (1,3) -> deg = [2, 3, 3, 2, 0]
đường (2,4) -> deg = [2, 3, 4, 2, 1]
  giá trị 1 -> đỉnh 4 (bậc 1): +  1  tổng = 1
  giá trị 2 -> đỉnh 0 (bậc 2): +  4  tổng = 5
  giá trị 3 -> đỉnh 3 (bậc 2): +  6  tổng = 11
  giá trị 4 -> đỉnh 1 (bậc 3): + 12  tổng = 23
  giá trị 5 -> đỉnh 2 (bậc 4): + 20  tổng = 43
kết quả: 43


### 8.2 Trace cách 3 (đếm phân phối) trên một đồ thị hình sao + đỉnh cô lập

`n = 7`, tâm 0 nối 1..4, thêm đường (5,1); đỉnh 6 cô lập.

In [15]:
n7, r7 = 7, [[0, 1], [0, 2], [0, 3], [0, 4], [5, 1]]
deg = degrees(n7, r7); print("deg:", deg)
cnt = [0] * n7
for d in deg: cnt[d] += 1
k = 0; total = 0
for b in range(n7):
    c = cnt[b]
    if c:
        part = b * (c * (2 * k + c + 1) // 2)
        print(f"bậc {b}: {c} đỉnh nhận giá trị {k + 1}..{k + c} -> +{part}")
        total += part; k += c
print("tổng:", total, "| brute:", brute(n7, r7), "| sort_degrees:", sort_degrees(n7, r7))

deg: [4, 2, 1, 1, 1, 1, 0]
bậc 0: 1 đỉnh nhận giá trị 1..1 -> +0
bậc 1: 4 đỉnh nhận giá trị 2..5 -> +14
bậc 2: 1 đỉnh nhận giá trị 6..6 -> +12
bậc 4: 1 đỉnh nhận giá trị 7..7 -> +28
tổng: 54 | brute: 54 | sort_degrees: 54


Đỉnh cô lập (bậc 0) nhận giá trị nhỏ nhất `1` và đóng góp 0 — không phí giá trị lớn cho nó. Tâm sao bậc 4 nhận `7` ⇒ góp 28.

## 9. Code nộp LeetCode

In [16]:
from typing import List

class Solution:
    def maximumImportance(self, n: int, roads: List[List[int]]) -> int:
        deg = [0] * n
        for a, b in roads:              # mỗi đường góp 1 vào bậc của cả hai đầu
            deg[a] += 1
            deg[b] += 1
        deg.sort()                      # bậc nhỏ nhận giá trị nhỏ, bậc lớn nhận giá trị lớn
        return sum(i * d for i, d in enumerate(deg, 1))

In [17]:
sol = Solution()
for n, roads, exp in examples:
    got = sol.maximumImportance(n, roads)
    print(got, "| kỳ vọng", exp, "| OK" if got == exp else "| SAI")

43 | kỳ vọng 43 | OK
20 | kỳ vọng 20 | OK


## 10. Vì sao thuật toán đúng

**Bổ đề 1 (đổi tổng).** Với mọi phép gán `val`: `Σ_{(a,b) ∈ roads} (val[a] + val[b]) = Σ_v deg[v] · val[v]`. Chứng minh: khai triển vế trái thành tổng các hạng tử `val[x]` với `x` là một đầu mút của một đường; hạng tử `val[v]` xuất hiện đúng `deg[v]` lần.

**Bổ đề 2 (hoán đổi không làm giảm).** Nếu `deg[u] ≤ deg[v]` và `val[u] > val[v]` thì đổi giá trị của `u` và `v` làm tổng thay đổi `Δ = (deg[v] − deg[u])(val[u] − val[v]) ≥ 0`.

**Định lý.** Gọi `π` là thứ tự đỉnh theo bậc tăng dần (hòa phá tùy ý) và `val*` gán giá trị `i` cho đỉnh thứ `i` trong `π`. Khi đó `val*` tối ưu.

*Chứng minh.* Lấy phép gán tối ưu bất kỳ `val`, gọi **nghịch thế** là cặp `(u, v)` với `u` đứng trước `v` trong `π` (nên `deg[u] ≤ deg[v]`) nhưng `val[u] > val[v]`. Nếu `val ≠ val*` thì tồn tại hai đỉnh có giá trị **liên tiếp** `x + 1` (đỉnh `u`) và `x` (đỉnh `v`) với `u` đứng trước `v` trong `π` — nếu không, giá trị tăng dần theo `π`, tức `val = val*`. Đổi hai giá trị này: theo bổ đề 2 tổng không giảm (nên vẫn tối ưu), và số nghịch thế giảm **đúng 1**, vì với mọi đỉnh thứ ba `w`, `val[w]` không nằm giữa `x` và `x + 1` nên quan hệ của `w` với `u` và `v` giữ nguyên. Số nghịch thế hữu hạn ⇒ sau hữu hạn bước thu được `val*` với tổng bằng tổng tối ưu ⇒ `val*` tối ưu. (Cell dưới chạy đúng quy trình này trên một phép gán ngẫu nhiên.)

**Kết luận:** đáp án = `Σ_{i=1..n} i · deg_sorted[i]`. ∎

**Bổ sung:** chặn trên của đáp án: `deg ≤ n − 1`, `Σ deg = 2m` ⇒ đáp án ≤ `n · 2m ≤ 5·10⁴ · 10⁵ = 5·10⁹` — vượt int32 (quan trọng trong C++/Java, không quan trọng trong Python).

In [18]:
# minh họa chứng minh: từ một phép gán bất kỳ, sửa dần các nghịch thế kề nhau về giá trị, tổng không giảm
rng = random.Random(10)
n, roads = 6, [[0, 1], [0, 2], [0, 3], [1, 2], [4, 5], [2, 4]]
deg = degrees(n, roads); key = sorted(range(n), key=lambda v: (deg[v], v)); pos = {v: i for i, v in enumerate(key)}
val = list(range(1, n + 1)); rng.shuffle(val)
print("bậc:", deg, "| gán ban đầu:", val, "tổng", importance(roads, val))
changed = True
while changed:
    changed = False
    who = {val[v]: v for v in range(n)}
    for x in range(1, n):
        u, v = who[x + 1], who[x]              # u có giá trị lớn hơn v đúng 1
        if pos[u] < pos[v]:                    # nhưng u đứng trước v trong thứ tự bậc
            before = importance(roads, val)
            val[u], val[v] = val[v], val[u]
            after = importance(roads, val)
            assert after >= before
            print(f"  đổi đỉnh {u}<->{v}: {before} -> {after}")
            changed = True; break
print("kết thúc:", val, "tổng", importance(roads, val), "| tối ưu:", Solution().maximumImportance(n, roads))

bậc: [3, 2, 3, 1, 2, 1] | gán ban đầu: [3, 6, 2, 4, 1, 5] tổng 38
  đổi đỉnh 0<->2: 38 -> 38
  đổi đỉnh 3<->2: 38 -> 40
  đổi đỉnh 3<->0: 40 -> 42
  đổi đỉnh 3<->4: 42 -> 43
  đổi đỉnh 5<->2: 43 -> 45
  đổi đỉnh 5<->0: 45 -> 47
  đổi đỉnh 5<->4: 47 -> 48
  đổi đỉnh 1<->2: 48 -> 49
  đổi đỉnh 1<->0: 49 -> 50
  đổi đỉnh 1<->4: 50 -> 50
kết thúc: [5, 3, 6, 1, 4, 2] tổng 50 | tối ưu: 50


## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn hoán vị trên cạnh | O(n! · m) | O(n) |
| 1b. Vét cạn trên dãy bậc | O(n! · n) | O(n) |
| 2. Sắp xếp bậc | O(m + n log n) | O(n) |
| 3. Đếm phân phối | O(m + n) | O(n) |
| 4. Heap | O(m + n log n) | O(n) |
| 5. Counter | O(m + n log n) | O(n) |

In [19]:
rng = random.Random(2285)
N = 5 * 10**4
edges = set()
while len(edges) < 5 * 10**4:
    a, b = rng.randrange(N), rng.randrange(N)
    if a != b: edges.add((min(a, b), max(a, b)))
big = [list(e) for e in edges]
star = [[0, i] for i in range(1, N)]
for label, rr in [("ngẫu nhiên", big), ("hình sao", star)]:
    res = []
    for name, f in [("sort_degrees", sort_degrees), ("counting", counting), ("heap_assign", heap_assign),
                    ("counter_oneline", counter_oneline), ("Solution", Solution().maximumImportance)]:
        t0 = time.perf_counter(); res.append(f(N, rr)); dt = time.perf_counter() - t0
        print(f"{label:<11} {name:<16} {dt*1000:7.1f} ms -> {res[-1]}")
    assert len(set(res)) == 1

ngẫu nhiên  sort_degrees         9.5 ms -> 3463492117
ngẫu nhiên  counting             7.6 ms -> 3463492117
ngẫu nhiên  heap_assign         14.9 ms -> 3463492117


ngẫu nhiên  counter_oneline     22.0 ms -> 3463492117


ngẫu nhiên  Solution             9.8 ms -> 3463492117
hình sao    sort_degrees         5.0 ms -> 3749925000
hình sao    counting             4.8 ms -> 3749925000


hình sao    heap_assign         11.5 ms -> 3749925000


hình sao    counter_oneline      9.1 ms -> 3749925000
hình sao    Solution             4.4 ms -> 3749925000


## 12. Bộ test

### 12.1 Ví dụ đề + edge case (đều đối chiếu với vét cạn)

In [20]:
ALL = {"brute": brute, "brute_deg": brute_deg, "sort_degrees": sort_degrees, "counting": counting,
       "heap_assign": heap_assign, "counter_oneline": counter_oneline, "Solution": Solution().maximumImportance}
edge = examples + [
    (2, [[0, 1]], 3),                                     # nhỏ nhất: 1 + 2
    (3, [[0, 1]], 5),                                     # một đỉnh cô lập nhận 1: 2 + 3
    (4, [[0, 1], [1, 2], [2, 3], [3, 0]], 20),            # chu trình: mọi bậc 2 -> 2·(1+2+3+4)
    (4, [[0, 1], [0, 2], [0, 3]], 18),                    # sao: tâm 4·3 + 1 + 2 + 3
    (4, [[0, 1], [0, 2], [0, 3], [1, 2], [1, 3], [2, 3]], 30),   # đồ thị đầy đủ K4: 3·10
    (6, [[0, 1], [2, 3], [4, 5]], 21),                    # ghép cặp hoàn hảo: 1+…+6
]
for n, roads, exp in edge:
    for name, f in ALL.items():
        got = f(n, roads)
        assert got == exp, (name, n, roads, got, exp)
    print(f"n={n} m={len(roads)} -> {exp}")
print("edge OK")

n=5 m=6 -> 43
n=5 m=3 -> 20
n=2 m=1 -> 3
n=3 m=1 -> 5
n=4 m=4 -> 20
n=4 m=3 -> 18
n=4 m=6 -> 30
n=6 m=3 -> 21
edge OK


### 12.2 Đối chiếu ngẫu nhiên với vét cạn (seed cố định)

In [21]:
def rand_graph(rng, n):
    all_e = [[a, b] for a in range(n) for b in range(a + 1, n)]
    m = rng.randint(1, len(all_e))
    return rng.sample(all_e, m)

rng = random.Random(42)
for _ in range(400):
    n = rng.randint(2, 7); roads = rand_graph(rng, n)
    ref = brute_deg(n, roads)
    for name, f in ALL.items():
        if name == "brute" and n > 6: continue
        assert f(n, roads) == ref, (name, n, roads)
print("400 đồ thị ngẫu nhiên OK")

400 đồ thị ngẫu nhiên OK


### 12.3 Tính chất: đáp án = `Σ deg · val*`; nằm giữa giá trị nhỏ nhất (ghép ngược) và `n · 2m`

In [22]:
rng = random.Random(7)
for _ in range(500):
    n = rng.randint(2, 30); roads = rand_graph(rng, n)
    ans = Solution().maximumImportance(n, roads)
    d = sorted(degrees(n, roads))
    worst = sum(i * v for i, v in enumerate(d[::-1], 1))        # ghép ngược: nhỏ nhất
    assert worst <= ans <= n * 2 * len(roads)
    assert importance(roads, assignment(n, roads)) == ans
    assert ans >= (n + 1) * len(roads)                          # trung bình 2 giá trị ngẫu nhiên là n+1
print("tính chất OK")

tính chất OK


Vì sao `ans ≥ (n + 1)·m`? Kỳ vọng tổng của một phép gán **ngẫu nhiên đều** là `Σ deg · (n+1)/2 = 2m · (n+1)/2 = (n+1)m`; tối ưu không thể nhỏ hơn trung bình.

### 12.4 Edge cực trị: đồ thị hình sao n = 5·10⁴ — kiểm tra công thức đóng

In [23]:
N = 5 * 10**4
star = [[0, i] for i in range(1, N)]
got = Solution().maximumImportance(N, star)
closed = N * (N - 1) + (N - 1) * N // 2 - 0            # tâm: N·(N−1); lá: giá trị 1..N−1, mỗi lá bậc 1
print(got, closed, got == closed, "| vượt int32?", got > 2**31 - 1)
assert got == closed

3749925000 3749925000 True | vượt int32? True


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Sắp xếp **giảm dần** nhưng vẫn gán `1, 2, …` (ghép ngược chiều)

In [24]:
def wrong_desc(n, roads):
    deg = sorted(degrees(n, roads), reverse=True)
    return sum(i * v for i, v in enumerate(deg, 1))            # SAI: bậc lớn nhận giá trị nhỏ

for n, roads, exp in examples:
    print("bản sai :", wrong_desc(n, roads), "| bản đúng:", Solution().maximumImportance(n, roads))

bản sai : 29 | bản đúng: 43
bản sai : 16 | bản đúng: 20


**Vì sao sai:** đây chính là **giá trị nhỏ nhất** theo bất đẳng thức sắp xếp lại. **Sửa:** cùng chiều — tăng dần với `1..n`, hoặc giảm dần với `n..1`.

### Lỗi 2 — Chỉ tăng bậc của một đầu mút

In [25]:
def wrong_one_end(n, roads):
    deg = [0] * n
    for a, b in roads: deg[a] += 1                             # SAI: quên deg[b]
    deg.sort()
    return sum(i * v for i, v in enumerate(deg, 1))

for n, roads, exp in examples:
    print("bản sai :", wrong_one_end(n, roads), "| bản đúng:", Solution().maximumImportance(n, roads))

bản sai : 24 | bản đúng: 43
bản sai : 12 | bản đúng: 20


**Sửa:** đường hai chiều, độ quan trọng cộng **cả hai** đầu ⇒ tăng cả `deg[a]` và `deg[b]`.

### Lỗi 3 — Dùng `Counter` mà quên đỉnh cô lập (gán giá trị bắt đầu từ 1 cho đỉnh có bậc)

In [26]:
def wrong_counter(n, roads):
    c = Counter(x for r in roads for x in r)
    return sum(i * v for i, v in enumerate(sorted(c.values()), 1))   # SAI: bỏ qua các đỉnh bậc 0

for n, roads in [(examples[0][0], examples[0][1]), (3, [[0, 1]]), (7, [[0, 1], [0, 2], [0, 3], [0, 4], [5, 1]])]:
    print("bản sai :", wrong_counter(n, roads), "| bản đúng:", Solution().maximumImportance(n, roads))

bản sai : 43 | bản đúng: 43
bản sai : 3 | bản đúng: 5
bản sai : 44 | bản đúng: 54


**Vì sao sai:** ví dụ 1 không có đỉnh cô lập nên **trùng hợp** đúng (dòng đầu). Với `n = 3, roads = [[0,1]]`, đỉnh 2 cô lập nên đỉnh có bậc phải nhận `2, 3` (tổng 5), nhưng bản sai gán `1, 2` (tổng 3). **Sửa:** `deg = [0] * n` hoặc bắt đầu `enumerate` từ `n − len(c) + 1`.

### Lỗi 4 — Gán theo **chỉ số thành phố** (`val[v] = v + 1`) thay vì theo bậc

In [27]:
def wrong_by_index(n, roads):
    return sum((a + 1) + (b + 1) for a, b in roads)            # SAI: không tối ưu gì cả

for n, roads, exp in examples:
    print("bản sai :", wrong_by_index(n, roads), "| bản đúng:", Solution().maximumImportance(n, roads))

bản sai : 33 | bản đúng: 43
bản sai : 19 | bản đúng: 20


### Lỗi 5 — Tham lam theo **cạnh**: duyệt cạnh, gán hai giá trị lớn nhất còn lại cho đầu mút chưa có giá trị

In [28]:
def wrong_edge_greedy(n, roads):
    val = [0] * n; nxt = n
    for a, b in roads:                                         # SAI: thứ tự cạnh quyết định, không theo bậc
        for v in (a, b):
            if val[v] == 0: val[v] = nxt; nxt -= 1
    for v in range(n):
        if val[v] == 0: val[v] = nxt; nxt -= 1
    return importance(roads, val)

for n, roads, exp in examples:
    print("bản sai :", wrong_edge_greedy(n, roads), "| bản đúng:", Solution().maximumImportance(n, roads))

bản sai : 39 | bản đúng: 43
bản sai : 19 | bản đúng: 20


**Vì sao sai:** đỉnh ở cạnh đầu tiên nhận giá trị lớn dù có thể chỉ có bậc 1. Tổng phụ thuộc vào **bậc**, không vào thứ tự xuất hiện. **Sửa:** tính bậc trước rồi mới gán.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Tổng trên cạnh của `f(đỉnh)`" ⇒ đổi thành **tổng trên đỉnh có trọng số bậc** (contribution technique).
- "Gán các giá trị khác nhau để tối đa/tối thiểu Σ hệ số × giá trị" ⇒ **bất đẳng thức sắp xếp lại**: sắp cả hai, ghép cùng chiều (max) hoặc ngược chiều (min).
- Chứng minh tham lam nhanh: viết `Δ` khi hoán đổi hai phần tử — nếu `Δ = (x₂ − x₁)(y₁ − y₂)` thì chắc chắn là sắp xếp lại.
- Khóa nhỏ (bậc ≤ n − 1) ⇒ có thể **đếm phân phối** O(n).
- Cẩn thận tràn số ở ngôn ngữ khác: đáp án tới ~`2,5·10⁹`.
- Bài cùng họ: 1029 Two City Scheduling, 1874 Minimize Product Sum of Two Arrays, 2279 Maximum Bags With Full Capacity of Rocks, 1561 Maximum Number of Coins You Can Get, 2333 Minimum Sum of Squared Difference.

## 15. Biến thể của bài

### 15.1 Biến thể A — Tổng **nhỏ nhất** (ghép ngược chiều)

In [29]:
def minimum_importance(n, roads):
    d = sorted(degrees(n, roads), reverse=True)
    return sum(i * v for i, v in enumerate(d, 1))

def brute_min(n, roads):
    d = degrees(n, roads)
    return min(sum(dv * pv for dv, pv in zip(d, p)) for p in permutations(range(1, n + 1)))

rng = random.Random(151)
for _ in range(200):
    n = rng.randint(2, 7); roads = rand_graph(rng, n)
    assert minimum_importance(n, roads) == brute_min(n, roads)
print(minimum_importance(*examples[0][:2]), "(nhỏ nhất cho ví dụ 1)")

29 (nhỏ nhất cho ví dụ 1)


### 15.2 Biến thể B — Đường có **trọng số** (độ quan trọng = `w · (val[a] + val[b])`), hoặc có đường trùng

Thay `deg` bằng **bậc có trọng số** `W[v] = Σ w` của các đường chạm `v`; đường trùng chỉ là trọng số cộng dồn. Tham lam y nguyên.

In [30]:
def weighted_importance(n, roads_w):
    W = [0] * n
    for a, b, w in roads_w:
        W[a] += w; W[b] += w
    W.sort()
    return sum(i * x for i, x in enumerate(W, 1))

def weighted_brute(n, roads_w):
    return max(sum(w * (p[a] + p[b]) for a, b, w in roads_w) for p in permutations(range(1, n + 1)))

rng = random.Random(152)
for _ in range(200):
    n = rng.randint(2, 6)
    rw = [[a, b, rng.randint(1, 9)] for a, b in rand_graph(rng, n)]
    rw += [rw[0]] if rng.random() < 0.3 else []                 # có thể lặp đường
    assert weighted_importance(n, rw) == weighted_brute(n, rw)
print(weighted_importance(3, [[0, 1, 5], [1, 2, 1], [0, 1, 2]]))

39


### 15.3 Biến thể C — Tập giá trị cho trước (không phải `1..n`)

Bất đẳng thức sắp xếp lại vẫn áp dụng: sắp `deg` và `values` cùng chiều.

In [31]:
def with_values(n, roads, values):
    return sum(d * v for d, v in zip(sorted(degrees(n, roads)), sorted(values)))

def with_values_brute(n, roads, values):
    d = degrees(n, roads)
    return max(sum(dv * pv for dv, pv in zip(d, p)) for p in permutations(values))

rng = random.Random(153)
for _ in range(200):
    n = rng.randint(2, 6); roads = rand_graph(rng, n); vals = [rng.randint(-10, 10) for _ in range(n)]
    assert with_values(n, roads, vals) == with_values_brute(n, roads, vals)
print(with_values(*examples[0][:2], [10, 0, 0, 0, 1]))

43


Giá trị **âm** cũng được: bất đẳng thức sắp xếp lại không cần số dương.

### 15.4 Biến thể D — Độ quan trọng là **tích** `val[a] · val[b]`: tham lam theo bậc **không còn đúng**

Hàm mục tiêu không tách được thành tổng theo đỉnh ⇒ cấu trúc đồ thị quan trọng. Ta tìm một phản ví dụ bằng vét cạn.

In [32]:
def product_brute(n, roads):
    return max(sum(p[a] * p[b] for a, b in roads) for p in permutations(range(1, n + 1)))

def product_greedy_by_degree(n, roads):
    val = assignment(n, roads)
    return sum(val[a] * val[b] for a, b in roads)

rng = random.Random(154); found = None
for _ in range(3000):
    n = rng.randint(3, 6); roads = rand_graph(rng, n)
    if product_greedy_by_degree(n, roads) != product_brute(n, roads):
        found = (n, roads); break
n_, r_ = found
print("phản ví dụ:", found, "| bậc:", degrees(n_, r_))
print("tham lam theo bậc:", product_greedy_by_degree(n_, r_), "| tối ưu thật:", product_brute(n_, r_))

phản ví dụ: (6, [[2, 4], [2, 3], [2, 5], [0, 1], [1, 4], [1, 5], [3, 4], [1, 3], [1, 2], [0, 5], [0, 4]]) | bậc: [3, 5, 4, 3, 4, 3]
tham lam theo bậc: 148 | tối ưu thật: 153


Bài học: tham lam theo bậc đúng **vì** tổng là **tuyến tính** theo giá trị của từng đỉnh. Khi các đỉnh tương tác (tích), bài trở thành bài gán phức tạp hơn nhiều (họ quadratic assignment, NP-khó tổng quát).

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn hoán vị | định nghĩa | O(n!·m) | O(n) | kiểm chứng n ≤ 8 |
| 2. Sắp xếp bậc | sắp xếp lại | O(m + n log n) | O(n) | **mặc định** |
| 3. Đếm phân phối | bậc ≤ n−1 | O(m + n) | O(n) | tối ưu tiệm cận |
| 4. Heap | phát giá trị lớn nhất | O(m + n log n) | O(n) | giá trị đến dần |
| 5. Counter | gọn | O(m + n log n) | O(n) | nhớ bù đỉnh cô lập |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `maximumImportance(3, [[0,1],[1,2]])` và `maximumImportance(5, [[0,1]])`?

**Bài 2 (tính tay):** `n = 6`, `roads = [[0,1],[0,2],[0,3],[3,4],[3,5],[1,2]]`. Tính dãy bậc, phép gán tối ưu và đáp án.

**Bài 3 (điền TODO):** Viết `bt3_todo(n, roads)` trả về **một phép gán tối ưu** `val` (danh sách độ dài `n`, là hoán vị của `1..n`).

**Bài 4 (tự viết lại):** Viết `bt4_todo(n, roads)` cài **cách đếm phân phối** O(n + m) không nhìn code.

**Bài 5 (mini challenge):** Viết `bt5_todo(n, roads)` trả về **số phép gán tối ưu khác nhau** (mod `10⁹ + 7`). Gợi ý: các đỉnh cùng bậc hoán đổi tự do; đỉnh khác bậc thì không.

In [33]:
def bt3_todo(n, roads):
    # TODO: trả về một phép gán tối ưu
    pass

def bt4_todo(n, roads):
    # TODO: đếm phân phối O(n + m)
    pass

def bt5_todo(n, roads):
    # TODO: số phép gán tối ưu mod 1e9+7
    raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: tính bậc. Mức 2: `[1, 2, 1]` và `[1, 1, 0, 0, 0]`. Mức 3: `1+2+2·3 = 9`; `4 + 5 = 9`.

**Bài 2.** Mức 1: `deg = [3, 2, 2, 3, 1, 1]`. Mức 2: sắp `[1,1,2,2,3,3]`. Mức 3: `1+2+6+8+15+18 = 50`.

**Bài 3.** Mức 1: thứ tự đỉnh theo bậc. Mức 2: `order = sorted(range(n), key=deg.__getitem__)`. Mức 3: `val[order[i]] = i + 1`.

**Bài 4.** Mức 1: `cnt[b]` = số đỉnh bậc `b`. Mức 2: khối giá trị `k+1..k+c`. Mức 3: cộng `b · c · (2k + c + 1) / 2`, `k += c`.

**Bài 5.** Mức 1: theo bổ đề hoán đổi, đỉnh bậc lớn hơn **thực sự** phải nhận giá trị lớn hơn — vì `Δ > 0` khi `deg[u] < deg[v]`. Mức 2: các nhóm cùng bậc nhận một khối giá trị cố định, hoán vị bên trong tùy ý. Mức 3: `Π cnt[b]!` mod `10⁹+7`.

## 19. Đáp án tham khảo cho bài tập

**Bài 1:** `9`, `9`. **Bài 2:** `50`.

In [34]:
assert Solution().maximumImportance(3, [[0, 1], [1, 2]]) == 9
assert Solution().maximumImportance(5, [[0, 1]]) == 9
assert Solution().maximumImportance(6, [[0, 1], [0, 2], [0, 3], [3, 4], [3, 5], [1, 2]]) == 50

def bt3(n, roads):
    deg = degrees(n, roads)
    order = sorted(range(n), key=deg.__getitem__)
    val = [0] * n
    for i, v in enumerate(order): val[v] = i + 1
    return val

def bt4(n, roads):
    return counting(n, roads)

def bt5(n, roads, MOD=10**9 + 7):
    cnt = Counter(degrees(n, roads)); res = 1
    for c in cnt.values():
        for k in range(2, c + 1): res = res * k % MOD
    return res

def bt5_brute(n, roads):
    best = brute_deg(n, roads); d = degrees(n, roads)
    return sum(1 for p in permutations(range(1, n + 1)) if sum(a * b for a, b in zip(d, p)) == best)

rng = random.Random(1985)
for _ in range(300):
    n = rng.randint(2, 6); roads = rand_graph(rng, n)
    v = bt3(n, roads)
    assert sorted(v) == list(range(1, n + 1)) and importance(roads, v) == brute_deg(n, roads)
    assert bt4(n, roads) == brute_deg(n, roads)
    assert bt5(n, roads) == bt5_brute(n, roads)
print(bt3(*examples[0][:2]), bt5(*examples[0][:2]), bt5(*examples[1][:2]))
print("Đáp án bài tập: tất cả assert OK")

[2, 4, 5, 3, 1] 2 24
Đáp án bài tập: tất cả assert OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
deg = [0] * n
for a, b in roads:
    deg[a] += 1; deg[b] += 1          # tổng theo cạnh = Σ deg[v] * val[v]
deg.sort()                            # sắp xếp lại: ghép cùng chiều
return sum(i * d for i, d in enumerate(deg, 1))
```

**Checklist:**
- [ ] Tăng bậc **cả hai** đầu mút.
- [ ] `deg = [0] * n` để gồm cả đỉnh cô lập (chúng nhận giá trị nhỏ nhất).
- [ ] Ghép **cùng chiều**: bậc tăng ↔ giá trị `1..n` tăng.
- [ ] (Ngôn ngữ khác) dùng số 64-bit: đáp án có thể vượt `2³¹`.